In [1]:
from osgeo import gdal
import numpy as np
import pandas as pd
#import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../Functions')
import TiffTools as tt

%load_ext autoreload
%autoreload 2

In [ ]:
tt.micmacExport('/Users/chanagan/Downloads/CarsonNevada04142026/M57_NW_Ortho_v2.tif', 
                outname='/Users/chanagan/Downloads/CarsonNevada04142026/mmM57_NW_Ortho_v2.tif', 
                srs='EPSG:32611', outres=[0.6,-0.6], interp=None, a_ullr=None,cutlineDSName=None,nodata=0)

Computing Gray from RGB values
Writing to /Users/chanagan/Downloads/CarsonNevada04142026/mmM57_NW_Ortho_v2.tif


In [ ]:
mm3d Mm2dPosSism 24APR06185248-P1BS-200012802885_01_P007_toAOI.tif \
    26MAY10190720-P1BS-200012802831_01_P001_toAOI.tif \
    CorMin=0.1 Dequant=false DirMEC='MEC_WV/' SsResolOpt=1

In [19]:
folder = '/home/chanagan/tmp/rotationTest/MEC/'
prefiles = ['/home/chanagan/tmp/rotationTest/NCALM2007_DropBox_translate_pos80cm_y_rotated_05292026.tif',
            '/home/chanagan/tmp/rotationTest/USGS_FEMA_2017_D18_merge_clipped_epsg6339_rotated.tif']

In [20]:
tt.micmacPostProcessing(folder=folder,
                         prefiles=prefiles,
                         outprefix=folder)

Nodata value for mask: -9999.0
Setting nodata value to -9999
Saving /home/chanagan/tmp/rotationTest/MEC/NSmicmac.tif
Saving /home/chanagan/tmp/rotationTest/MEC/EWmicmac.tif
Saving /home/chanagan/tmp/rotationTest/MEC/Correlmicmac.tif


In [6]:
par, perp = tt.projectDisp(folder+'MEC100_50/EWmicmac.tif',folder+'MEC100_50/NSmicmac.tif',315,partif=folder+'MEC100_50/ParallelDisp.tif',perptif=folder+'MEC100_50/PerpendicularDisp.tif')

In [5]:
tt.createMicmacParamFile('mm09AUG04191333.tif','mm13MAR11192237.tif',results_directory='MEC100_50/',SzW=100,CorrelMin=0.1,SzW_base=50)

'./ param_LeChantier_Compl.xml written.'

# Raster Rotation


In [39]:
import math
import os
import numpy as np
import rasterio

from affine import Affine
from rasterio.warp import reproject
from rasterio.enums import Resampling

files = [
    '/home/chanagan/tmp/rotationTest/MEC/EWmicmac.tif',
    '/home/chanagan/tmp/rotationTest/MEC/NSmicmac.tif'
]

angle = -41  # clockwise

for file in files:

    with rasterio.open(file) as src:

        data = src.read()

        h = src.height
        w = src.width

        # Original transform
        T = src.transform

        # Raster center in pixel coords
        cx = w / 2
        cy = h / 2

        # Rotation in pixel space
        R = (
            Affine.translation(-cx, -cy)
            * Affine.rotation(angle)
            * Affine.translation(cx, cy)
        )

        # Transform corners
        corners = [
            (0, 0),
            (w, 0),
            (w, h),
            (0, h)
        ]

        rotated = [R * pt for pt in corners]

        xs = [p[0] for p in rotated]
        ys = [p[1] for p in rotated]

        minx = min(xs)
        maxx = max(xs)
        miny = min(ys)
        maxy = max(ys)

        new_w = int(math.ceil(maxx - minx))
        new_h = int(math.ceil(maxy - miny))

        # Shift so rotated raster fits entirely
        shifted = (
            Affine.translation(-minx, -miny)
            * R
        )

        dst_transform = T * shifted

        dst_data = np.zeros(
            (src.count, new_h, new_w),
            dtype=data.dtype
        )

        reproject(
            source=data,
            destination=dst_data,

            src_transform=src.transform,
            src_crs=src.crs,

            dst_transform=dst_transform,
            dst_crs=src.crs,

            resampling=Resampling.nearest
        )

        profile = src.profile.copy()

        profile.update(
            width=new_w,
            height=new_h,
            transform=dst_transform
        )

        outfile = os.path.splitext(file)[0] + "_rotated.tif"

        with rasterio.open(outfile, "w", **profile) as dst:
            dst.write(dst_data)

In [41]:
import rioxarray
from scipy.ndimage import rotate
files = [
    '/home/chanagan/tmp/rotationTest/MEC/EWmicmac.tif',
    '/home/chanagan/tmp/rotationTest/MEC/NSmicmac.tif'
]

angle = -41  # azimuth CW from north/y

for file in files:
    da = rioxarray.open_rasterio(file)

    rot = rotate(da.values, -41, axes=(1,2), reshape=False)

    da.values = rot

    da.rio.to_raster(file[:-4]+"_rotated.tif")

In [47]:
import numpy as np
import rasterio
from scipy.ndimage import rotate

files = [
    '/home/chanagan/tmp/rotationTest/MEC/EWmicmac.tif',
    '/home/chanagan/tmp/rotationTest/MEC/NSmicmac.tif'
]

angle = 41

for file in files:

    out_file = file.replace(".tif", "_rotated.tif")

    with rasterio.open(file) as src:

        data = src.read(1)
        nodata = src.nodata

        # If nodata is not defined, choose a fallback (optional)
        if nodata is None:
            nodata = np.nan

        # Build mask
        if np.isnan(nodata):
            mask = np.isnan(data)
        else:
            mask = (data == nodata)

        # Replace nodata with 0 (or any neutral value) before rotation
        data_filled = np.where(mask, 0, data).astype(data.dtype)

        # Rotate data and mask
        rotated_data = rotate(
            data_filled,
            angle=angle,
            reshape=True,
            order=0,
            mode='constant',
            cval=0
        )

        rotated_mask = rotate(
            mask.astype(np.uint8),
            angle=angle,
            reshape=True,
            order=0,
            mode='constant',
            cval=1  # outside original bounds becomes masked
        )

        # Re-apply nodata
        rotated = rotated_data.astype(data.dtype)
        rotated[rotated_mask == 1] = nodata

        profile = src.profile.copy()
        profile.update({
            'height': rotated.shape[0],
            'width': rotated.shape[1],
            'nodata': nodata
        })

    with rasterio.open(out_file, 'w', **profile) as dst:
        dst.write(rotated, 1)

    print(f"Saved: {out_file}")

Saved: /home/chanagan/tmp/rotationTest/MEC/EWmicmac_rotated.tif
Saved: /home/chanagan/tmp/rotationTest/MEC/NSmicmac_rotated.tif
